# Stage 1 Kaggle Run-All Notebook

This notebook provides a single Kaggle/Linux execution path for Stage 1:
- bootstrap and runtime guards
- dataset attach-path verification
- hardware profile selection (P100 or T4x2)
- Stage 1 training (first run)
- Stage 1 resume flow after session interruption
- decode sanity validation and artifact discovery

All dataset access follows Kaggle attach-only paths under `/kaggle/input`, and outputs are written under `/kaggle/working`.

## 1) Parameters: Hardware + Run Mode

Set these values before running all cells.

In [ ]:
# Parameters
HARDWARE_PROFILE = "p100"   # options: "p100" or "t4x2"
RUN_MODE = "first"          # options: "first" or "resume"

# Dataset contract (attach in Kaggle UI first)
DATASET_SLUG = "balraj98/modelnet40-princeton-3d-object-dataset"
DATASET_ROOT_OVERRIDE = ""   # optional: explicit /kaggle/input path if Kaggle uses a different folder name

# Optional: explicit resume checkpoint.
# Leave empty to use default /kaggle/working/checkpoints/latest_step.ckpt in resume mode.
RESUME_CHECKPOINT_PATH = ""

# Decode sanity parameters
DECODE_NUM_SAMPLES = 4
DECODE_THRESHOLD = 0.5

In [ ]:
import os
from pathlib import Path

def _looks_like_modelnet_root(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    if any(path.rglob('*.off')):
        return True
    for subdir in ('ModelNet40', 'modelnet40', 'ModelNet40_Aligned'):
        candidate = path / subdir
        if candidate.exists() and candidate.is_dir() and any(candidate.rglob('*.off')):
            return True
    return False


def _resolve_dataset_root(dataset_slug: str, explicit_root: str = '') -> Path:
    input_root = Path('/kaggle/input')
    expected_name = dataset_slug.split('/')[-1]
    checked = []

    for candidate in [explicit_root, os.environ.get('DATASET_ROOT', '')]:
        candidate = str(candidate).strip()
        if candidate:
            checked.append(Path(candidate))

    checked.extend([
        input_root / expected_name,
        input_root / dataset_slug.replace('/', '-'),
    ])

    if input_root.exists():
        checked.extend([entry for entry in sorted(input_root.iterdir()) if entry.is_dir()])

    unique_candidates = []
    seen = set()
    for candidate in checked:
        resolved = candidate if candidate.is_absolute() else (input_root / candidate)
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        unique_candidates.append(resolved)

    existing = [candidate for candidate in unique_candidates if candidate.exists() and candidate.is_dir()]
    for candidate in existing:
        if _looks_like_modelnet_root(candidate):
            return candidate

    meaningful_existing = [candidate for candidate in existing if candidate != input_root]
    if len(meaningful_existing) == 1:
        return meaningful_existing[0]

    mounted = [entry.name for entry in sorted(input_root.iterdir()) if entry.is_dir()] if input_root.exists() else []
    mounted_label = mounted or ['none']
    raise RuntimeError(
        'Dataset root missing. Attach the Kaggle dataset in the notebook UI before training. '
        f'Checked candidates: {[str(candidate) for candidate in unique_candidates]}. '
        f'Mounted /kaggle/input folders: {mounted_label}'
    )


ROOT = Path('/kaggle/working/DoAn_backup')
if not ROOT.exists():
    ROOT = Path.cwd()

os.chdir(ROOT)
print(f'Project root: {ROOT}')

hardware_map = {
    'p100': 'configs/hardware_p100.yaml',
    't4x2': 'configs/hardware_t4x2.yaml',
}
if HARDWARE_PROFILE not in hardware_map:
    raise ValueError(f'Invalid HARDWARE_PROFILE: {HARDWARE_PROFILE}')

if RUN_MODE not in {'first', 'resume'}:
    raise ValueError(f'Invalid RUN_MODE: {RUN_MODE}')

hardware_cfg = hardware_map[HARDWARE_PROFILE]
dataset_root = _resolve_dataset_root(DATASET_SLUG, DATASET_ROOT_OVERRIDE)

env = {
    'DATASET_SLUG': DATASET_SLUG,
    'DATASET_ROOT': str(dataset_root),
    'OUTPUT_ROOT': '/kaggle/working',
    'TRAIN_CONFIG': 'configs/train_stage1.yaml',
    'DATA_CONFIG': 'configs/data_stage1.yaml',
    'HARDWARE_CONFIG': hardware_cfg,
}

for key, value in env.items():
    os.environ[key] = value

resume_default = '/kaggle/working/checkpoints/latest_step.ckpt'
if RUN_MODE == 'resume':
    os.environ['RESUME_CHECKPOINT_PATH'] = RESUME_CHECKPOINT_PATH or resume_default

print('Resolved environment:')
for key in ['DATASET_SLUG', 'DATASET_ROOT', 'OUTPUT_ROOT', 'TRAIN_CONFIG', 'DATA_CONFIG', 'HARDWARE_CONFIG']:
    print(f'  {key}={os.environ[key]}')
if RUN_MODE == 'resume':
    print(f"  RESUME_CHECKPOINT_PATH={os.environ['RESUME_CHECKPOINT_PATH']}")

## 2) Bootstrap + Runtime Guards

This runs deterministic package setup and fail-fast checks for:
- GPU/CUDA visibility
- free disk in `/kaggle/working`
- writable output/checkpoint/log directories
- dataset root under `/kaggle/input/<dataset-slug>`

It also captures metadata and config snapshot under `/kaggle/working/runs/<run_id>/metadata/`.

In [ ]:
!bash scripts/kaggle_bootstrap.sh

In [ ]:
from pathlib import Path

dataset_root = Path(os.environ['DATASET_ROOT'])
output_root = Path(os.environ['OUTPUT_ROOT'])

print(f'Dataset root exists: {dataset_root.exists()} -> {dataset_root}')
print(f'Output root exists:  {output_root.exists()} -> {output_root}')

if not dataset_root.exists():
    raise RuntimeError(
        'Dataset root missing. Attach dataset in Kaggle UI (Add data) before training.'
    )

## 3) Stage 1 Training (First Run vs Resume)

- **First run** uses `--autoresume --contract-smoke` and starts from scratch if no checkpoint exists.
- **Resume run** uses the same path plus explicit `--resume-from` (default `latest_step.ckpt`) after session interruption.

Both flows use the canonical trainer: `scripts/train_stage1.py` -> `src/train/train_stage1.py`.

In [ ]:
import subprocess

base_cmd = [
    'python', 'scripts/train_stage1.py',
    '--config', os.environ['TRAIN_CONFIG'],
    '--hardware', os.environ['HARDWARE_CONFIG'],
    '--data-config', os.environ['DATA_CONFIG'],
    '--dataset-root', os.environ['DATASET_ROOT'],
    '--output-root', os.environ['OUTPUT_ROOT'],
    '--autoresume',
    '--contract-smoke',
    '--device', 'auto',
]

if RUN_MODE == 'resume':
    resume_path = os.environ.get('RESUME_CHECKPOINT_PATH', '/kaggle/working/checkpoints/latest_step.ckpt')
    cmd = base_cmd + ['--resume-from', resume_path]
    print('Running RESUME flow:')
else:
    cmd = base_cmd
    print('Running FIRST-RUN flow:')

print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

checkpoint_dir = Path('/kaggle/working/checkpoints')
log_dir = Path('/kaggle/working/logs')

print('Checkpoint candidates:')
for name in ['best.ckpt', 'latest.ckpt', 'latest_step.ckpt', 'interrupt.ckpt']:
    p = checkpoint_dir / name
    print(f'  {name}: {p.exists()}')

metrics_path = log_dir / 'stage1_training_metrics.jsonl'
print(f'\nMetrics path: {metrics_path} (exists={metrics_path.exists()})')

runs_root = Path('/kaggle/working/runs')
run_summaries = sorted(runs_root.glob('*/metadata/train_summary.json')) if runs_root.exists() else []
if run_summaries:
    latest_summary = run_summaries[-1]
    print(f'Latest train summary: {latest_summary}')
    print(latest_summary.read_text()[:1200])

## 4) Decode Sanity Validation (Kaggle/Linux Authoritative)

This runs the canonical decode sanity runner: `src/inference/generate_mesh.py`.

It evaluates available checkpoint candidates in this order:
1. `best.ckpt`
2. `latest.ckpt`
3. `latest_step.ckpt`
4. `interrupt.ckpt`

Reports are written to:
- `/kaggle/working/logs/stage1_decode_sanity_report.json`
- `/kaggle/working/logs/stage1_decode_sanity_report.md`
- Mesh exports under `/kaggle/working/inference_decode_sanity/`

In [ ]:
decode_cmd = [
    'python', 'src/inference/generate_mesh.py',
    '--config', os.environ['TRAIN_CONFIG'],
    '--data-config', os.environ['DATA_CONFIG'],
    '--output-root', os.environ['OUTPUT_ROOT'],
    '--num-samples', str(DECODE_NUM_SAMPLES),
    '--threshold', str(DECODE_THRESHOLD),
    '--device', 'auto',
]
print(' '.join(decode_cmd))
subprocess.run(decode_cmd, check=True)

In [ ]:
import json
from pathlib import Path

report_json = Path('/kaggle/working/logs/stage1_decode_sanity_report.json')
report_md = Path('/kaggle/working/logs/stage1_decode_sanity_report.md')

print(f'JSON report: {report_json} (exists={report_json.exists()})')
print(f'MD report:   {report_md} (exists={report_md.exists()})')

if report_json.exists():
    payload = json.loads(report_json.read_text())
    print('\nDecode summary:')
    print(json.dumps(payload.get('decode_summary', {}), indent=2))
    print('\nPreflight errors:')
    print(payload.get('preflight_errors', []))

## 5) Artifact Listing + Export Guidance

All run artifacts are under `/kaggle/working`:
- checkpoints: `/kaggle/working/checkpoints`
- logs and decode reports: `/kaggle/working/logs`
- run metadata snapshots: `/kaggle/working/runs/<run_id>/metadata`
- decode mesh exports: `/kaggle/working/inference_decode_sanity`

Publish options after training:
1. Download artifacts directly from Kaggle notebook output files.
2. Create a Kaggle Dataset version from selected `/kaggle/working` artifacts.

In [ ]:
from pathlib import Path

targets = [
    Path('/kaggle/working/checkpoints'),
    Path('/kaggle/working/logs'),
    Path('/kaggle/working/runs'),
    Path('/kaggle/working/inference_decode_sanity'),
]

for t in targets:
    print(f'\n[{t}] exists={t.exists()}')
    if t.exists():
        for p in sorted(t.rglob('*'))[:40]:
            if p.is_file():
                print('  ', p)